# 🎵 MoodBeats Lab 2: Can the Computer Find the Moods on Its Own?
### A Catholic Introduction to Artificial Intelligence — Unsupervised Learning with k-Means

In the first MoodBeats lab, **you gave the computer the answer key.** Every song came with a `mood` label (Happy, Sad, Relaxed, Motivated), and the model learned to copy those labels. That is **supervised learning**: a teacher supervises by supplying the right answers.

Today we take the answer key away.

**k-means clustering** is a kind of **unsupervised learning**. It gets the songs' sound measurements and nothing else. Its only job is to sort the songs into *k* groups so that songs in the same group are as close together as possible. It doesn't know what a "mood" is. It doesn't even know it's looking at music.

**The big question:** If the computer sorts the songs by sound alone, will its groups match the moods humans assigned?

### How k-means works (in four steps)
1. Drop *k* random "center points" into the data.
2. Assign every song to its nearest center.
3. Move each center to the average position of the songs assigned to it.
4. Repeat steps 2–3 until the centers stop moving.

That's the whole algorithm. Everything it "knows" comes from one idea: **distance**.

### What you'll do
| Step | What happens |
|---|---|
| 1 | Load and clean the data |
| 2 | Hide the answer key |
| 3 | Experiment 1: cluster *without* scaling (and watch it go wrong) |
| 4 | Experiment 2: cluster *with* scaling |
| 5 | Meet your clusters, and **name them yourself** |
| 6 | How many groups are there, really? |
| 7 | The Reveal: compare clusters to moods |
| 8 | Where did "Motivated" go? |
| 9 | Picture it in 2D |
| 10 | Reflection |

> 💡 Put `spotify_tracks_mood.csv` in the same folder as this notebook before you start. Everything here runs on a regular laptop with no GPU.

## INSTRUCTIONS:

- Be sure all cells run without error.

- Where you see this ---> ✍️ Your Observations, write down your observations. Be sure to follow the rules from the Student Writing Guidelines and use numbers from the notebook to support your answer.

## Step 0: Setup

# Install the necessary libraries if you haven't already
%pip install pandas numpy scikit-learn matplotlib seaborn

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA

pd.set_option("display.precision", 2)
SEED = 42   # keeps everyone's results the same
print("Ready!")

## Step 1: Load and Clean the Data

In [ ]:
df = pd.read_csv("spotify_tracks_mood.csv")
print("Rows and columns:", df.shape)
df.head()

Some songs appear more than once (the same track listed on several albums or playlists). Each has the same `id`, so we keep one copy of each. We also drop songs with a `tempo` of 0, which means the measurement failed (often silence or a spoken track).

In [ ]:
before = len(df)
df = df.drop_duplicates(subset="id")
after_dupes = len(df)
df = df[df["tempo"] > 0].reset_index(drop=True)

print(f"Started with:            {before:,} rows")
print(f"Removed duplicates:      {before - after_dupes:,}")
print(f"Removed tempo = 0:       {after_dupes - len(df):,}")
print(f"Songs left to cluster:   {len(df):,}")

In [ ]:
print("How the human-assigned moods are spread out:")
print(df["mood"].value_counts())

## Step 2: Hide the Answer Key 🙈

We give k-means only measurements of **how each song sounds**. We leave out:

- **`mood`**: the answer key. That's the whole point.
- **`explicit`, `mode`**: on/off (0/1) columns. k-means measures straight-line distance, and a jump from 0 to 1 doesn't mean "a little more" of anything.
- **`key`, `time_signature`**: these are numbers, but they're really *categories*. Key 11 (B) isn't "more" than key 1 (C♯).
- **`popularity`, `duration_ms`**: facts *about* the song, not its sound.

In [ ]:
SOUND_FEATURES = ["danceability", "energy", "loudness", "speechiness",
                  "acousticness", "instrumentalness", "liveness",
                  "valence", "tempo"]

X = df[SOUND_FEATURES]          # what k-means gets to see
answer_key = df["mood"]         # locked away until Step 7

X.describe().loc[["min", "max", "mean"]]

### ✍️ Your Observations
*Look at the min and max of each column. Which columns have much bigger numbers than the others? Why might that matter for an algorithm that measures distance?*

> _Double-click this cell and type your answer here._

## Step 3: Experiment 1 — Cluster *Without* Scaling

Let's ask for 4 groups (one for each mood) and see what happens if we just hand k-means the raw numbers.

In [ ]:
km_raw = KMeans(n_clusters=4, n_init=10, random_state=SEED)
raw_labels = km_raw.fit_predict(X)

# The average song in each cluster
pd.DataFrame(km_raw.cluster_centers_, columns=SOUND_FEATURES).round(2)

### ✍️ Your Observations
*Compare the four rows. Which column changes a lot from cluster to cluster? Which columns barely change at all? What do you think k-means actually sorted the songs by?*

> _Double-click this cell and type your answer here. Be sure to follow the course's Student Writing Guidelines and use numbers from the notebook to support your answer._

## Step 4: Experiment 2 — Cluster *With* Scaling

**Scaling** puts every column on the same footing. `StandardScaler` converts each value into "how many steps above or below average is this song?" (statisticians call these steps *standard deviations*). After scaling, a big jump in tempo and a big jump in energy count the same.

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

pd.DataFrame(X_scaled, columns=SOUND_FEATURES).describe().loc[["min", "max", "mean"]]

In [ ]:
km = KMeans(n_clusters=4, n_init=10, random_state=SEED)
raw_numbers = km.fit_predict(X_scaled)

# k-means numbers its clusters 0-3 in no particular order.
# We rename them A, B, C, D from largest to smallest so everyone's labels match.
order = pd.Series(raw_numbers).value_counts().index
rename = {old: letter for old, letter in zip(order, "ABCD")}
df["cluster"] = pd.Series(raw_numbers).map(rename)

print("Songs in each cluster:")
print(df["cluster"].value_counts().sort_index())

## Step 5: Meet Your Clusters 👋

Here is the average song in each cluster, back in the original units so you can read them.

In [ ]:
profile = df.groupby("cluster")[SOUND_FEATURES].mean()
profile

Raw averages are hard to compare (is a loudness of −15 a lot?). The chart below shows each cluster in **scaled units**: blue means below the average song, red means above, and the number says by how many steps.

In [ ]:
z_profile = pd.DataFrame(X_scaled, columns=SOUND_FEATURES).groupby(df["cluster"]).mean()

fig, ax = plt.subplots(figsize=(10, 3.6))
im = ax.imshow(z_profile.values, cmap="RdBu_r", vmin=-2, vmax=2, aspect="auto")
ax.set_xticks(range(len(SOUND_FEATURES)), SOUND_FEATURES, rotation=35, ha="right")
ax.set_yticks(range(len(z_profile)), [f"Cluster {c}" for c in z_profile.index])
for i in range(z_profile.shape[0]):
    for j in range(z_profile.shape[1]):
        v = z_profile.values[i, j]
        ax.text(j, i, f"{v:+.1f}", ha="center", va="center",
                color="white" if abs(v) > 1.2 else "black", fontsize=9)
fig.colorbar(im, ax=ax, label="steps from average song")
ax.set_title("What makes each cluster different?")
plt.tight_layout()
plt.show()

### ✍️ Your turn: Name the clusters (before you peek!)

k-means gave us letters, not names. **Naming is a human job.** Using the chart above, write a short name for each cluster in the cell below, like "Quiet Acoustic Songs" or "Loud Dance Tracks." Don't use the four mood words yet.  Which categories are above average?  Below average? Answer: `Songs in this cluster are..."`

In [ ]:
my_names = {
    "A": "???",
    "B": "???",
    "C": "???",
    "D": "???",
}
my_names

### ✍️ Your Observations
*Which cluster was easiest to name? Which was hardest? Did any cluster surprise you?*

> _Double-click this cell and type your answer here. Be sure to follow the course's Student Writing Guidelines and use numbers from the notebook to support your answer._

## Step 6: How Many Groups Are There, Really?

We asked for 4 clusters because there are 4 moods. But the music didn't tell us that; *we* did. Let's ask k-means for 2 through 8 clusters and score each try two ways:

- **Inertia** (left chart): the total distance from each song to its cluster's center. Lower is tighter. It *always* drops as you add clusters, so we look for an **"elbow"** where adding more stops helping much.
- **Silhouette score** (right chart): roughly, "Is each song much closer to its own cluster than to the next-nearest one?" It runs from −1 to 1, and higher means cleaner, more separate groups. (We score a random sample of 5,000 songs to keep it fast.)

In [ ]:
ks = range(2, 9)
inertias, silhouettes = [], []
sample = np.random.RandomState(SEED).choice(len(X_scaled), 5000, replace=False)

for k in ks:
    model = KMeans(n_clusters=k, n_init=10, random_state=SEED).fit(X_scaled)
    inertias.append(model.inertia_)
    silhouettes.append(silhouette_score(X_scaled[sample], model.labels_[sample]))
    print(f"k = {k}:  inertia = {model.inertia_:>9,.0f}   silhouette = {silhouettes[-1]:.3f}")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.6))
ax1.plot(list(ks), inertias, "o-", color="#1f3a68")
ax1.set(title="Inertia (look for an elbow)", xlabel="number of clusters k", ylabel="inertia")
ax2.plot(list(ks), silhouettes, "o-", color="#b8860b")
ax2.set(title="Silhouette (higher = cleaner groups)", xlabel="number of clusters k", ylabel="silhouette score")
plt.tight_layout()
plt.show()

### ✍️ Your Observations

Left Chart: What's important here are the number of clusters (k): how many clusters before the curve flattens out (called the **elbow**). Right Chart:  Higher is better, so you want the highest point, not a bend.

*Is there a clear elbow? Which k gets the best silhouette score? Does the data seem to "want" 4 groups?*

> _Double-click this cell and type your answer here. Be sure to follow the course's Student Writing Guidelines and use numbers from the notebook to support your answer._

## Step 7: The Reveal 🎭

Time to unlock the answer key. This table counts how many songs of each mood landed in each cluster.

In [ ]:
reveal = pd.crosstab(df["cluster"], answer_key)
reveal

Raw counts are hard to compare because Happy songs outnumber everything else. Here is the same table as **percentages of each cluster**.

In [ ]:
reveal_pct = pd.crosstab(df["cluster"], answer_key, normalize="index") * 100

fig, ax = plt.subplots(figsize=(7, 3.6))
im = ax.imshow(reveal_pct.values, cmap="Blues", vmin=0, vmax=100, aspect="auto")
ax.set_xticks(range(reveal_pct.shape[1]), reveal_pct.columns)
ax.set_yticks(range(reveal_pct.shape[0]), [f"Cluster {c}" for c in reveal_pct.index])
for i in range(reveal_pct.shape[0]):
    for j in range(reveal_pct.shape[1]):
        v = reveal_pct.values[i, j]
        ax.text(j, i, f"{v:.0f}%", ha="center", va="center",
                color="white" if v > 60 else "black")
ax.set_title("What % of each cluster has each mood?")
ax.set_xlabel("human-assigned mood")
plt.tight_layout()
plt.show()

print("Biggest mood in each cluster:")
for c in reveal_pct.index:
    top = reveal_pct.loc[c].idxmax()
    print(f"  Cluster {c}: {reveal_pct.loc[c, top]:.0f}% {top}")

### ✍️ Your Observations
*Which cluster matches a single mood best? Which moods never get a cluster of their own? Compare these results to the names you chose in Step 5.*

> _Double-click this cell and type your answer here. Be sure to follow the course's Student Writing Guidelines and use numbers from the notebook to support your answer._

## Step 8: Where Did "Motivated" Go? 🕵️

In the first MoodBeats lab we noticed something odd about the `explicit` column. Let's look again: what share of explicit and non-explicit songs got each mood?

In [ ]:
pd.crosstab(df["explicit"].map({0: "not explicit", 1: "explicit"}),
            answer_key, normalize="index").mul(100).round(1)

### Experiment 3: Let k-means see the `explicit` column
We left `explicit` out in Step 2 because it isn't a sound measurement. What happens if we add it back?

In [ ]:
X_plus = StandardScaler().fit_transform(df[SOUND_FEATURES + ["explicit"]])
plus_labels = KMeans(n_clusters=4, n_init=10, random_state=SEED).fit_predict(X_plus)

order = pd.Series(plus_labels).value_counts().index
plus_clusters = pd.Series(plus_labels).map({old: l for old, l in zip(order, "ABCD")})
pd.crosstab(plus_clusters.rename("cluster"), answer_key)

### ✍️ Your Observations
*What happened once k-means could see `explicit`? What does this tell you about how the "Motivated" label was probably made? Is "Motivated" really a sound?*

> _Double-click this cell and type your answer here. Be sure to follow the course's Student Writing Guidelines and use numbers from the notebook to support your answer._

## Step 9: Picture It in 2D

We can't draw 9 dimensions. **PCA** (Principal Component Analysis) squashes all 9 features down to the 2 directions where the songs spread out the most, so we can make a flat map. Some detail is lost, but nearby dots are still similar songs. We plot a random sample of 3,000 songs.

In [ ]:
pca = PCA(n_components=2, random_state=SEED)
coords = pca.fit_transform(X_scaled)
print(f"These 2 directions keep {pca.explained_variance_ratio_.sum():.0%} of the spread in the data.")

pick = np.random.RandomState(SEED).choice(len(coords), 3000, replace=False)
cluster_colors = {"A": "#1f77b4", "B": "#ff7f0e", "C": "#2ca02c", "D": "#d62728"}
mood_colors = {"Happy": "#f2b701", "Sad": "#3969ac", "Relaxed": "#11a579", "Motivated": "#e73f74"}

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5), sharex=True, sharey=True)
for c, col in cluster_colors.items():
    m = (df["cluster"].values[pick] == c)
    ax1.scatter(coords[pick][m, 0], coords[pick][m, 1], s=6, alpha=0.5, color=col, label=f"Cluster {c}")
for mood, col in mood_colors.items():
    m = (answer_key.values[pick] == mood)
    ax2.scatter(coords[pick][m, 0], coords[pick][m, 1], s=6, alpha=0.5, color=col, label=mood)
ax1.set(title="Colored by k-means cluster", xlabel="PCA direction 1", ylabel="PCA direction 2")
ax2.set(title="Colored by human-assigned mood", xlabel="PCA direction 1")
for ax in (ax1, ax2):
    ax.legend(markerscale=3, loc="upper right")
plt.tight_layout()
plt.show()

### ✍️ Your Observations
*Compare the two maps. Where do they agree? Where do the moods overlap so much that no straight-line boundary could separate them?*

> _Double-click this cell and type your answer here. Be sure to follow the course's Student Writing Guidelines and use numbers from the notebook to support your answer._

## 🧪 Try It Yourself
Change one thing, re-run, and write down what happened:
1. In Step 4, change `n_clusters=4` to `2` or `6`. Re-run Steps 4, 5 and 7.
2. Remove `liveness` from `SOUND_FEATURES` and re-run from Step 2.
3. Change `SEED` to another number and re-run Step 4.

## Step 10: Reflection ✝️

### Part 1
Answer all of the questions in this part in 3-5 sentences each. Use specific numbers from your notebook!

1. When you changed `n_clusters=4` to `2` or `6` and re-ran Steps 4, 5 and 7, did the new clusters line up with moods better or worse? Why?

> _Type your answer here. Be sure to follow the course's Student Writing Guidelines and use numbers from the notebook to support your answer._

2. When you removed `liveness` from `SOUND_FEATURES` and re-ran from Step 2, which cluster disappears? Why?

> _Type your answer here. Be sure to follow the course's Student Writing Guidelines and use numbers from the notebook to support your answer._

3. When you changed `SEED` to another number and re-ran Step 4, do the cluster sizes change much? Why?

> _Type your answer here. Be sure to follow the course's Student Writing Guidelines and use numbers from the notebook to support your answer._

### Part 2
Answer **one** of the following questions in this part in 3-5 sentences each. Use specific numbers from your notebook!

1. **Who names the groups?** k-means produced letters, not names. You named the clusters in Step 5. When companies cluster *people* (shoppers, voters, students, patients), someone names those groups too: "high-risk," "low-value," "likely to quit." How does the Church's teaching that each person is made in the image of God (*Imago Dei*) challenge the idea that a person *is* the group an algorithm put them in?

> _Type your answer here. Be sure to follow the course's Student Writing Guidelines and use numbers from the notebook to support your answer._

2. **Are the categories real?** The data never "asked" for 4 groups, and "Motivated" turned out to be mostly a lyrics flag rather than a sound. What does this suggest about labels and categories that people build into a dataset? Who should be responsible for checking them?

> _Type your answer here. Be sure to follow the course's Student Writing Guidelines and use numbers from the notebook to support your answer._

3. **The technocratic paradigm.** Pope Francis and Pope Leo XIV warn against treating reality as only what can be measured and calculated. A song's mood is something people *experience*. What can sound measurements capture about music, and what do they miss?

> _Type your answer here. Be sure to follow the course's Student Writing Guidelines and use numbers from the notebook to support your answer._


© 2026 D. E. Goddu. All rights reserved.
Developed by the author with the assistance of AI tools (Claude, Anthropic). All content was selected, reviewed, and edited by the author.